# Day 2





# Notebook 2 — Exploratory Data Analysis & Visualization (Matplotlib & Seaborn)

**Duration:** 75 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** visualize target and feature distributions, identify relationships, and produce publication-quality diagnostic plots.

## Learning objectives

- Plot distributions and summary statistics of the target variable.
- Visualize pairwise relationships and compute correlation matrices (Pearson and Spearman).
- Use violin plots and 2D density plots to inspect conditional distributions.
- Save figures for use in reports and presentations.

---

## Part 1 — Distribution analysis

### 1.1 Load the cleaned data and imports



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')

# CLEANED_PATH = Path(os.environ.get('SCRATCH', '.')) / 'Fe-Redox-GNN' / 'data_cleaned.csv'
CLEANED_PATH = OUTPUT_DIR +'/data_cleaned.pkl'
df = pd.read_pickle(CLEANED_PATH)
print('Loaded cleaned data with', len(df), 'rows from', CLEANED_PATH)
display(df.head())
fe_features


### 1.2 Target distribution and moments

Plot the distribution of the redox target and compute skewness and kurtosis. The sample skewness and (excess) kurtosis are defined as:

$$
\text{skew}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{3}\right], \quad \text{kurt}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{4}\right] - 3
$$



In [ ]:
TARGET_COL = 'reduction_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f'{TARGET_COL} column not found in cleaned data')

plt.figure(figsize=(6,4))
sns.histplot(df[TARGET_COL], kde=True)
plt.title('Reduction potential distribution')
plt.savefig('redox_distribution.png', dpi=150)
plt.show()

print('skewness:', df[TARGET_COL].skew())
print('kurtosis (excess):', df[TARGET_COL].kurt())



### 1.3 Feature distributions

Visualize a small set of numeric features with histograms and KDEs. This helps spot multimodality and heavy tails.



In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['index', TARGET_COL]]

def plot_feature_distributions(df, cols):
    n = len(cols)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, c in zip(axes, cols):
        sns.histplot(df[c].dropna(), kde=True, ax=ax)
        ax.set_title(c)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    fig.tight_layout()
    fig.savefig('feature_distributions.png', dpi=150)
    plt.show()

plot_feature_distributions(df, num_cols[:8])



---

## Part 2 — Relationship analysis

### 2.1 Scatter plots and Pearson correlation

Pearson correlation coefficient between two variables is defined as:

$$
r_{XY} = \frac{\sum_{i} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i} (x_i - \bar{x})^2}\ \sqrt{\sum_{i} (y_i - \bar{y})^2}}
$$

Plot a selection of features against the target and overlay a linear regression fit.



In [ ]:
pairs = [p for p in ['n_O', 'n_ligands', 'n_anionic'] if p in df.columns]
if not pairs:
    pairs = num_cols[:3]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4), squeeze=False)
axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, pairs):
    sns.regplot(x=c, y=TARGET_COL, data=df, ax=ax, scatter_kws={'s':10}, line_kws={'color':'red'})
    ax.set_title(f'{c} vs {TARGET_COL}')
for ax in axes[len(pairs):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig('feature_vs_target_scatter.png', dpi=150)
plt.show()

for c in pairs:
    r = df[c].corr(df[TARGET_COL])
    print(f'Pearson r ({c} vs {TARGET_COL}):', r)



### 2.2 Pairplot for multi-feature relationships

A pairplot (scatter + KDE on diag) helps identify non-linear relationships and clusters.



In [ ]:
cols_for_pairplot = [c for c in ['n_O', 'n_ligands', 'n_anionic', 'num_atoms', 'q', TARGET_COL] if c in df.columns]
if len(cols_for_pairplot) > 1:
    sns.pairplot(df[cols_for_pairplot].dropna(), diag_kind='kde', corner=True)
    plt.savefig('pairplot.png', dpi=150)
    plt.show()



---

## Part 3 — Correlation analysis

### 3.1 Correlation matrix heatmap

Compute Pearson correlation matrix and plot a heatmap.



In [ ]:
corr = df[num_cols + [TARGET_COL]].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Pearson correlation matrix')
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()



### 3.2 Spearman rank correlation

Spearman rank correlation is useful for monotonic but non-linear relationships. The rank correlation coefficient is:

$$
\rho = 1 - \frac{6 \sum_{i} d_i^2}{n(n^2 - 1)}\quad\text{where } d_i = \mathrm{rank}(x_i) - \mathrm{rank}(y_i)
$$

Compute and compare Spearman to Pearson for the target relationships.



In [ ]:
spearman = df[num_cols + [TARGET_COL]].corr(method='spearman')
print(f'Top correlations with {TARGET_COL} (Pearson):')
print(corr[TARGET_COL].abs().sort_values(ascending=False).head(10))
print(f'\nTop correlations with {TARGET_COL} (Spearman):')
print(spearman[TARGET_COL].abs().sort_values(ascending=False).head(10))

# q and q_sum encode almost the same charge information. Keep one for modeling.
if 'q_sum' in num_cols:
    num_cols.remove('q_sum')
    print('\nDropping q_sum from model features because it duplicates q.')



---

## Part 4 — Advanced visualizations

### 4.1 Violin plots by category

If a categorical variable exists (e.g., `ligand_type` or `solvent`), violin plots show conditional distributions of the target.



In [ ]:
# cat = None
# for c in df.select_dtypes(include=['object', 'category']).columns:
#     if df[c].nunique() < 10:
#         cat = c
#         break

cat_cols = None

cat_cols = df.select_dtypes(include=['object','category' ]).columns.tolist()
cat_cols.remove('desolv_atoms')
cat_cols.remove('tmqm_atoms')
cat_cols.remove('atoms')
cat_cols
for c in cat_cols: 
     print (df[c].nunique())
     if df[c].nunique() < 10:
        cat = c
        break   

if cat:
    plt.figure(figsize=(8,4))
    sns.violinplot(x=cat, y=TARGET_COL, data=df)
    plt.xticks(rotation=45)
    plt.title(f'Redox by {cat}')
    plt.savefig('violin_plots.png', dpi=150)
    plt.show()
else:
    print('No low-cardinality categorical column found for violin plots')



### 4.2 2D density plot

A 2D kernel density estimate lets you visualize joint distributions and identify dense regions where models should focus their fit.



In [ ]:
if 'n_O' in df.columns and 'n_ligands' in df.columns:
    plt.figure(figsize=(6,5))
    sns.kdeplot(x=df['n_O'], y=df['n_ligands'], cmap='Blues', fill=True, thresh=0.05)
    plt.xlabel('n_O')
    plt.ylabel('n_l_density.png', dpi=150)
    plt.show()



**Mentor checkpoint 4**

- Confirm whether the observed correlations align with chemical intuition.
- Discuss which features are plausible inputs for classical baselines (RF, GPR) vs. GNN inputs.
- Decide on a shortlist of features to use in Day 3 baselines.

Proceed only after confirmation.

---



### Exercise 2.1 — Custom correlation heatmap (20 minutes)

Create a custom heatmap focusing on the top 10 features most correlated (absolute Pearson r) with `redox_potential`. Save it as `exercise_2_1_heatmap.png`.

Provide code that computes the top features, recomputes the correlation submatrix, and plots it with annotations.

### Exercise 2.2 — Feature relationship deep dive (25 minutes)

Pick one feature (for example `fe_bond_mean`) and explore polynomial relationships to the redox target. Fit polynomial models of degree 1..5 and report validation R² for each.

Coefficient of determination (R²):

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

Hint: Split the data into a train/validation split (e.g., 80/20) or use k-fold cross-validation to compare polynomial degrees fairly and avoid overfitting.

### Exercise 2.3 — Discussion (5 minutes)

Answer in Markdown: based on the EDA, what modeling approach would you try first and why? Consider model complexity, interpretability, and dataset size.

---



## Summary

| Visualization | Purpose | Library |
|---|---:|---|
| Histogram + KDE | Inspect target distribution, skewness | seaborn |
| Scatter + regplot | Inspect linear trends with target | seaborn |
| Pairplot | Multi-feature relationships | seaborn |
| Correlation heatmap | Global linear associations | seaborn, matplotlib |
| Violin / KDE / 2D density | Conditional distributions and joint density | seaborn |

---
